# Automated Wavelength Solution

[Chapter 10](10-wavelength-solution-interactive.ipynb) needed a person. Somebody looked at an arc
spectrum next to a reference figure, picked six lines, and fitted the solution from them. That
is fine at a terminal and impossible in a pipeline reducing three hundred frames overnight.

A pipeline has something the interactive session did not: **prior knowledge of the solution**.
The wavelength solution of a spectrograph depends on the instrument setup, the grism, the slit and
the detector, and on a stable, well-characterized instrument it changes little from night to
night. Once a set of arcs from a configuration has been calibrated and checked, the wavelength at the
reference pixel, the dispersion there and the curvature are known to within a small range, and
every later arc in that configuration can be calibrated by searching inside it.

`fit_dispersion` is that search. Instead of being told a few correct pairs, it looks for the
polynomial coefficients that put the observed lines closest to catalog lines, using differential
evolution over a bounded parameter space and a KD-tree to find each line's nearest catalog
neighbor cheaply. The output is the same `WavelengthSolution1D`. What it needs are the **bounds**:
a range for the wavelength at the reference pixel, a range for the dispersion there, and limits on
the higher-order coefficients, all bracketing the prior.

This is the workflow for pipelines on well-tested, well-behaved instruments, and the chapter is
written as one: derive the bounds from a validated solution, fit, and check the result.

In [ ]:
import sys
import warnings

sys.path.append("../src")

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.modeling import models
from specreduce.background import Background
from specreduce.extract import BoxcarExtract
from specreduce.tracing import FitTrace, FlatTrace
from specreduce.tilt_solution import TiltSolution
from specreduce.wavecal1d import WavelengthCalibration1D
from specreduce.wavesol1d import WavelengthSolution1D

from common import (
    GTC_HGAR,
    TILT_SOLUTION_FILE,
    WAVELENGTH_SOLUTION_FILE,
    fit_tilt_solution,
    fit_wavelength_solution,
    read_data,
)

plt.rc("figure", figsize=(8, 2.4), dpi=110)
plt.rc("font", size=9)

arcs, lamps, obj = read_data()
nx = obj.data.shape[1]
pixels = np.arange(float(nx))

## Setup

The inputs are exactly Chapter 10's: arc spectra extracted from frames rectified with Chapter 6's
stored tilt solution, and the same line lists. As there, the calibration class detects the lines
itself with `find_lines`. Every fit below builds its own instance, so that no run inherits
another's matches.

In [ ]:
reference_row = FlatTrace(arcs[0], 135)

if not TILT_SOLUTION_FILE.exists():  # normally written by Chapter 6
    fit_tilt_solution(arcs).to_asdf(TILT_SOLUTION_FILE)

solution = TiltSolution.from_asdf(TILT_SOLUTION_FILE)

rectified_arcs = [solution.resample(frame) for frame in arcs]
rectified_obj = solution.resample(obj)


spectra = [
    BoxcarExtract(frame, reference_row, width=15).spectrum for frame in rectified_arcs
]

## The prior

In a pipeline the prior is a configuration entry per instrument setup: the wavelength at the
reference pixel, the dispersion there and the higher-order terms, with a range around each, drawn
from a set of calibrations of that setup that somebody has checked. The set is what gives the
ranges their meaning, since the spread of tested solutions across nights is a measurement of how
much the instrument drifts. This chapter has one validated solution rather than a set, the one
Chapter 10 wrote to an ASDF file, so that file stands in as the prior and the ranges around it
are set by judgment. The guard below refits it if the file is missing, so the chapter runs on its
own.

The solution is a polynomial in `pixel - ref_pixel`, so its coefficients are directly the
quantities `fit_dispersion` asks about: the wavelength at the reference pixel, the dispersion
there, and the higher-order terms.

In [ ]:
if not WAVELENGTH_SOLUTION_FILE.exists():  # normally written by Chapter 10
    fit_wavelength_solution(arcs, solution).to_asdf(WAVELENGTH_SOLUTION_FILE)

prior = WavelengthSolution1D.from_asdf(WAVELENGTH_SOLUTION_FILE)
prior_axis = prior.pix_to_wav(pixels)
c0, c1, c2, c3 = prior.p2w[1].parameters

print("prior (Chapter 10's solution)")
print(f"  wavelength at pixel 512:  {c0:.2f} A")
print(f"  dispersion there:         {c1:.4f} A/pix")
print(f"  quadratic, cubic terms:   {c2:.3e}, {c3:.3e}")
print(f"  coverage:                 {prior_axis[0]:.0f} to {prior_axis[-1]:.0f} A")

The bounds follow from those numbers. How wide to make them is a judgment about how much the
instrument can drift between the validated calibration and tonight's arc, and the next section
says how much slack the search tolerates; the values below are comfortable for this instrument.

The higher-order limits are set from the prior as well, at three times its coefficients. They
must not be left unset: `fit_dispersion` then invents limits of its own, as
$\pm10^{\log_{10}(\bar{d}) - 2i}$ for coefficient $i$ with $\bar{d}$ the mean of the dispersion
bounds, and on this instrument the automatic quadratic limit, ±5.3 × 10⁻⁴, is tighter than the
prior's 8.8 × 10⁻⁴. With the answer outside the search box, no seed and no population size can
find it.

In [ ]:
config = dict(
    wavelength_bounds=(c0 - 150, c0 + 150),
    dispersion_bounds=(c1 - 0.2, c1 + 0.2),
    higher_order_limits=[3 * abs(c2), 3 * abs(c3)],
    degree=3,
)

## The automated fit

The global search is stochastic. `fit_dispersion` takes a `seed`, handed to
`differential_evolution`, and the same seed gives the same solution while different seeds give
slightly different ones, which is what makes the search repeatable in a pipeline and testable
here. Building the calibration, detecting the lines and fitting is three calls. Below them come
the numbers a pipeline can compute for itself, the rms, the matched line count and the coverage,
and one it cannot: the largest deviation of the fitted axis from the prior, available here only
because the prior is a validated solution for this very arc.

In [ ]:
calibration = WavelengthCalibration1D(
    arc_spectra=spectra,
    line_lists=[GTC_HGAR, ["NeI"], ["XeI"]],
    line_list_bounds=(5000, 10500),
    ref_pixel=512,
    unit=u.angstrom,
    wave_air=True,
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # unconverged refits on non-line detections
    calibration.find_lines(fwhm=2.5, noise_factor=15)
    calibration.fit_dispersion(seed=0, **config)

total_detected = sum(len(lines) for lines in calibration.observed_lines)

matched = sum(
    np.ma.asarray(lines).compressed().size
    for lines in calibration.observed_line_locations
)
# Freeze the matched set before reading the rms, so that the number describes
# the lines counted above.
calibration.remove_unmatched_lines()
axis = calibration.solution.pix_to_wav(pixels)
deviation = np.max(np.abs(axis - prior_axis))

print(
    f"rms {calibration.rms('wavelength'):.2f} A, {matched} of {total_detected} lines matched, "
    f"coverage {axis[0]:.0f} to {axis[-1]:.0f} A"
)
print(f"largest deviation from the prior: {deviation:.2f} A ({deviation / c1:.2f} pix)")

The fit lands on the prior to within a fraction of a pixel, from bounds alone and without a
single identified line. Given a search space that contains the answer, the global search finds it.

Which of the bounds does the work is worth knowing, and it is not the one intuition picks. The
information in the prior sits mostly in the curvature terms. With the higher-order limits kept at
three times the prior's coefficients, the wavelength bound can be opened to ±1000 Å and the
dispersion bound to ±1 Å per pixel and every seed still converges, because once the curvature is
pinned the line pattern itself determines the zero point and the scale. Loosen the higher-order
limits instead and convergence becomes a matter of the seed at ten times the prior's coefficients
and is lost at a hundred times: the curvature terms are what let a wrong solution lay the
detections smoothly onto the wrong catalog lines, and bounding them tightly removes those
solutions from the search space before it starts. Degree 3 and the default population size of 30
are stable on these arcs; a higher degree adds a coefficient to the search and buys nothing.

## A pipeline-shaped function

Putting it together: the class's own line detection, bounds derived from the validated solution,
an explicit degree and limits, a seed for reproducibility, and a final `refine_fit` at the
interactive chapter's tolerance. One thing a real pipeline adds, and this book does not, is an
automated check that the fit converged: a global search that lands on a wrong solution still
reports a plausible rms, computed over the lines it matched, and nobody is looking at the plot.
The matched line count and the fitted coverage are the natural inputs to such a check.

In [ ]:
def calibrate(arc_spectra, line_lists, config, seed=0):
    """Fit a wavelength solution automatically from prior-derived bounds."""
    calibration = WavelengthCalibration1D(
        arc_spectra=arc_spectra,
        line_lists=line_lists,
        line_list_bounds=(5000, 10500),
        ref_pixel=512,
        unit=u.angstrom,
        wave_air=True,
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        calibration.find_lines(fwhm=2.5, noise_factor=15)
        calibration.fit_dispersion(seed=seed, **config)
        calibration.refine_fit(max_match_distance=2.0)
    calibration.remove_unmatched_lines()
    return calibration


result = calibrate(spectra, [GTC_HGAR, ["NeI"], ["XeI"]], config)
axis = result.solution.pix_to_wav(pixels)
deviation = np.max(np.abs(axis - prior_axis))

matched = sum(len(lines) for lines in result.observed_lines)
print(
    f"rms {result.rms('wavelength'):.3f} A over {matched} lines, "
    f"coverage {axis[0]:.0f}-{axis[-1]:.0f} A"
)
print(
    f"agreement with the interactive solution: {deviation:.2f} A ({deviation / c1:.2f} pix)"
)

In [ ]:
fig, axs = plt.subplots(
    2, figsize=(8, 4.0), constrained_layout=True, height_ratios=[2, 1], sharex="all"
)

trace = FitTrace(
    rectified_obj, bins=32, peak_method="gaussian", trace_model=models.Chebyshev1D(4)
)
subtracted = Background.two_sided(
    rectified_obj, trace, separation=60, width=20
).sub_image()
science = BoxcarExtract(subtracted, trace, width=11).spectrum
calibrated = result.solution.resample(science)

axs[0].plot(calibrated.spectral_axis.value, calibrated.flux.value, lw=0.6)
axs[0].set_ylabel(f"Flux [{calibrated.flux.unit.to_string('latex_inline')}]")

axs[1].axhline(0, c="0.6", lw=0.8)
axs[1].plot(axis, axis - prior_axis, lw=0.9, c="C1")
axs[1].set(
    xlim=(axis[0], axis[-1]),
    xlabel=r"Wavelength [$\AA$]",
    ylabel="automated $-$\ninteractive [$\\AA$]",
);

The top panel is the science spectrum on the automated solution's wavelength axis; the bottom is
the difference between the two chapters' solutions.

That difference is at most 0.73 Å, an eighth of a pixel, and it is worth understanding why it is
not zero. Both chapters end with the same `refine_fit(max_match_distance=2.0)` over the same
detections and the same catalog, but they arrive there from different starting solutions, the
global search on one side and the anchors on the other. Each start hands the 2 Å window a slightly
different set of matched lines at the margin, the lines whose residuals sit near the tolerance,
and a least-squares fit to slightly different pairs gives a slightly different polynomial. The
disagreement is smaller than either solution's own rms, and it is the size of the ambiguity the
line list leaves. The two workflows are two ways into the same solution to within that.

## Summary

- Automated calibration is for **pipelines on stable, well-characterized instruments**. It
  starts from a prior on the solution, drawn from a set of checked calibrations of the same setup
  whose spread sets the bounds; here Chapter 10's single stored solution stands in for the set.
- **`fit_dispersion` searches inside bounds instead of starting from anchors.** It needs
  `wavelength_bounds` and `dispersion_bounds` at the reference pixel and `higher_order_limits`,
  all bracketing the prior, and returns the same `WavelengthSolution1D`.
- **Always set `higher_order_limits` explicitly.** The automatic default can be tighter than the
  true coefficients, and with the answer outside the search box no seed and no population size
  can find it.
- **The prior's value is in the curvature terms.** With the higher-order limits at three times
  the validated coefficients, the wavelength and dispersion bounds can be opened wide without
  losing convergence; loosen the higher-order limits instead and the search fails. Degree 3 and
  the default population size are stable here.
- Pass a `seed` to `fit_dispersion` to make a run reproducible.
- End with the same `refine_fit` as the interactive chapter. Given bounds that contain the answer,
  the automated solution then **agrees with the interactive one to an eighth of a pixel**, the
  residual difference being the ambiguity the line list leaves at the matching tolerance.
- A pipeline needs an **automated check that the fit converged**, which this book does not
  build. A wrong solution from the global search still reports a plausible rms over the lines it
  matched, so the matched line count and the fitted coverage belong in that check.

Next: [The 2D Wavelength Solution](12-wavelength-solution-2d.ipynb), which chains this solution
with the tilt solution into one object describing the whole detector.